# CLIPFORGE · Kostenloser iPhone-Live-Clipper (manueller Test)
**Ziel:** Einen öffentlich zugänglichen, zur Aufnahme freigegebenen Twitch-Livestream ca. 90–150 Sekunden aufnehmen, eine lebhafte Szene auswählen, eine vertikale **MP4-Datei** erzeugen und direkt aufs iPhone herunterladen.

- **Kosten:** Google Colab kann kostenlos genutzt werden, hat aber schwankende Limits. Kein PC und keine Kreditkarte für die kostenlose Colab-Nutzung.
- **Keine Dauerautomatik:** Manuell auf „Alle ausführen“ tippen; Notebook benötigt aktive Colab-Sitzung. Keine garantierte Aufnahme, wenn Twitch nicht live oder aus Colab nicht erreichbar ist.
- **Nur mit Erlaubnis:** Du brauchst Rechte zum Aufnehmen, Bearbeiten und Weiterveröffentlichen, einschließlich eventuell vorhandener Musik. Keine Umgehung von Twitch-Sperren oder DRM.
- **Privatsphäre:** Keine Zugangsdaten nötig. Dateien bleiben vorübergehend in der Colab-Sitzung; anschließend lokal herunterladen. Das Notebook lädt **nichts** selbstständig auf TikTok hoch.
- **Qualität:** 9:16-Video mit vollständig sichtbarem Gameplay und unscharfem Hintergrund. Highlight-Erkennung erfolgt über Lautstärke, nicht über inhaltliche KI.


In [ ]:
!python -m pip -q install "streamlink>=8,<9"
!ffmpeg -hide_banner -version | head -n 1


## Streamer wählen und Aufnahme erlauben
Wähle den Twitch-Benutzernamen und bestätige ausdrücklich, dass du die Rechte an Aufzeichnung, Bearbeitung und Weiterverbreitung hast. Colab-Formulare sind auf dem iPhone über das kleine Regler-/Bearbeiten-Symbol in der Zelle verfügbar.

In [ ]:
# @title Clip-Einstellungen
STREAMER = "xqc" # @param {type:"string"}
AUFNAHME_SEKUNDEN = 100 # @param {type:"slider", min:40, max:160, step:10}
CLIP_SEKUNDEN = 35 # @param {type:"slider", min:15, max:60, step:5}
BREITE = 720 # @param [480, 720, 1080] {type:"raw"}
NUTZUNGSRECHTE_BESTAETIGT = False # @param {type:"boolean"}


In [ ]:
import os, re, shutil, subprocess, time, datetime
from pathlib import Path

if not NUTZUNGSRECHTE_BESTAETIGT:
    raise PermissionError("Bitte nur bei tatsächlicher Aufnahme- und TikTok-Nutzungsfreigabe starten und Checkbox setzen.")
name = STREAMER.strip().lstrip("@").lower()
if not re.fullmatch(r"[a-z0-9_]{3,25}", name):
    raise ValueError("Ungültiger Twitch-Benutzername.")
if not shutil.which("streamlink") or not shutil.which("ffmpeg"):
    raise RuntimeError("Streamlink oder FFmpeg fehlt – bitte zuerst Installationszelle ausführen.")
capture_seconds = max(40, min(160, int(AUFNAHME_SEKUNDEN)))
clip_seconds = max(15, min(60, int(CLIP_SEKUNDEN)))
width = int(BREITE)
assert width in [480,720,1080]
folder = Path("/content/clipforge")
folder.mkdir(exist_ok=True)
raw = folder / "live_source.mkv"
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
output = folder / f"clipforge_{name}_{stamp}.mp4"

print("Prüfe öffentlichen Twitch-Livestream:", name)
status=subprocess.run(["streamlink", "--stream-url", f"https://www.twitch.tv/{name}", "best"],
                      capture_output=True, text=True, timeout=35)
if status.returncode != 0 or not status.stdout.strip().startswith("http"):
    raise RuntimeError("Kein öffentlich erreichbarer Live-Stream über Streamlink. Kanal kann offline, gesperrt oder aus Colab nicht erreichbar sein.")

print("Aufnahme gestartet. Bitte Colab-Sitzung offen lassen:", capture_seconds, "Sekunden")
source=subprocess.Popen(["streamlink", "--stdout", f"https://www.twitch.tv/{name}", "best"],
                        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
try:
    ffmpeg=subprocess.Popen(["ffmpeg", "-hide_banner", "-loglevel","error","-y",
        "-i","pipe:0","-t",str(capture_seconds),"-map","0:v:0","-map","0:a?",
        "-c","copy",str(raw)],
        stdin=source.stdout,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)
    source.stdout.close()
    try:
        _,err=ffmpeg.communicate(timeout=capture_seconds+55)
    except subprocess.TimeoutExpired:
        ffmpeg.kill()
        ffmpeg.communicate()
        raise RuntimeError("Aufnahme überschritt das Zeitlimit.")
    if ffmpeg.returncode != 0:
        raise RuntimeError("Live-Aufnahme fehlgeschlagen: "+err.decode(errors="replace")[-350:])
finally:
    if source.poll() is None:
        source.terminate()
        try: source.wait(timeout=5)
        except subprocess.TimeoutExpired:
            source.kill()
            source.wait()
if not raw.exists() or raw.stat().st_size<20000:
    raise RuntimeError("Stream liefert keine brauchbaren Videodaten.")
print("Aufnahmegröße:", round(raw.stat().st_size/1e6,1),"MB")


In [ ]:
import json
dur=subprocess.run(["ffprobe","-v","error","-show_entries","format=duration",
    "-of","default=noprint_wrappers=1:nokey=1",str(raw)],
    capture_output=True,text=True,timeout=15,check=True)
duration=float(dur.stdout.strip())
if duration<15: raise RuntimeError("Aufnahme ist zu kurz.")
clip_length=min(float(clip_seconds),duration-0.5)
max_start=max(0,duration-clip_length-0.5)
candidates=[round(max_start*i/5,1) for i in range(6)]
scores=[]
for start in candidates:
    measured=subprocess.run(["ffmpeg","-hide_banner","-ss",str(start),"-t",str(min(18,clip_length)),
        "-i",str(raw),"-vn","-af","volumedetect","-f","null","-"],
        capture_output=True,text=True,timeout=30)
    found=re.search(r"mean_volume:\s*(-?[\d.]+)\s*dB",measured.stderr)
    scores.append((float(found.group(1)) if found else -100,start))
score,start=max(scores)
print(f"Clip-Ausschnitt: {start:.1f}–{start+clip_length:.1f} s; Audio-Heuristik: {score:.1f} dB")

height=(round(width*16/9)//2)*2
filt=(
"[0:v]split=2[bg][fg];"
f"[bg]scale={width}:{height}:force_original_aspect_ratio=increase,crop={width}:{height},boxblur=20:1[back];"
f"[fg]scale={width}:{height}:force_original_aspect_ratio=decrease[front];"
"[back][front]overlay=(W-w)/2:(H-h)/2,format=yuv420p[v]"
)
cmd=["ffmpeg","-hide_banner","-loglevel","error","-y","-ss",str(start),
    "-i",str(raw),"-t",str(clip_length),"-filter_complex",filt,
    "-map","[v]","-map","0:a?","-r","30","-c:v","libx264",
    "-preset","veryfast","-crf","22","-pix_fmt","yuv420p",
    "-c:a","aac","-b:a","160k","-movflags","+faststart",str(output)]
render=subprocess.run(cmd,capture_output=True,text=True,timeout=300)
if render.returncode!=0 or not output.exists() or output.stat().st_size<10000:
    raise RuntimeError("MP4-Render fehlgeschlagen: "+render.stderr[-600:])
print("✅ Fertiger MP4-Clip:",output.name)
print("Größe:",round(output.stat().st_size/1e6,1),"MB","Auflösung:",width,"×",height)
raw.unlink(missing_ok=True)


In [ ]:
# MP4 direkt auf das iPhone laden. Falls Safari blockiert, in Colab
# links das Ordner-Symbol öffnen und über die drei Punkte bei der MP4 herunterladen.
from google.colab import files
print("Download startet:", output.name)
files.download(str(output))


### Falls nichts heruntergeladen wird
Links in Colab auf **Dateien / Ordner** tippen → Ordner `clipforge` öffnen → drei Punkte an der MP4 → **Herunterladen**. Unter iOS erscheint die Datei je nach Browser in **Dateien → Downloads**. Dies ist ein manueller Gratis-Test, keine ständig laufende Clip-Automatik.
